In [0]:
# dependencies
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DecimalType

In [0]:
# import csv data
VOLUME_PATH = "/Volumes/workspace/retail_fresher/retail_raw"
raw_customers = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/customers.csv")
raw_products = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/products.csv")
raw_sales_orders = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/sales_orders.csv")

In [0]:
# add source_file and ingestion_timestamp columns, save bronze tables
bronze_customers = (
    raw_customers
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/customers.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_products = (
    raw_products
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/products.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_sales_orders = (
    raw_sales_orders
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/sales_orders.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_customers")
bronze_products.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_products")
bronze_sales_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_sales_orders")

In [0]:
# load bronze tables for silver
customers = spark.read.table("workspace.retail_fresher.bronze_customers")
products = spark.read.table("workspace.retail_fresher.bronze_products")
sales_orders = spark.read.table("workspace.retail_fresher.bronze_sales_orders")

In [0]:
# customers silver process
customers = (
    customers
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.initcap(F.trim(F.col("customer_name"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn(
        "city",
        F.when(F.col("city").isNull() | (F.col("city") == ""), "Unknown").otherwise(F.col("city"))
    )
    .withColumn("state", F.initcap(F.trim(F.col("state"))))
    .withColumn("region", F.initcap(F.trim(F.col("region"))))
    .withColumn("customer_segment", F.initcap(F.trim(F.col("customer_segment"))))
    .withColumn("is_active", F.upper(F.trim(F.col("is_active"))))
    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd"))
    .withColumn("date_of_birth", F.to_date(F.col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("updated_at", F.to_timestamp(F.col("updated_at"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("loyalty_points", F.col("loyalty_points").try_cast(IntegerType()))
    .drop("source_file", "ingestion_timestamp")
)

w = Window.partitionBy("customer_id").orderBy(F.desc_nulls_last("updated_at"))

deduped_customers = (
    customers
    .withColumn("row_num", F.row_number().over(w))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .filter(F.col("is_active") == F.lit("Y"))
)

In [0]:
# products silver process
products = (
    products
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .withColumn("subcategory", F.initcap(F.trim(F.col("subcategory"))))
    .withColumn("supplier_name", F.initcap(F.trim(F.col("supplier_name"))))
    .withColumn("active_flag", F.upper(F.trim(F.col("active_flag"))))
    .withColumn("unit_price", F.col("unit_price").try_cast(DecimalType(10, 2)))
    .withColumn("cost_price", F.col("cost_price").try_cast(DecimalType(10, 2)))
    .withColumn("stock_quantity", F.col("stock_quantity").try_cast(IntegerType()))
    .withColumn("launch_date", F.to_date(F.col("launch_date"), "yyyy-MM-dd"))
    .withColumn("product_rating", F.col("product_rating").try_cast(DecimalType(3, 2)))
    .drop("source_file", "ingestion_timestamp")
    .filter(
        (F.col("unit_price") > 0) &
        (F.col("cost_price") > 0)
    )
    .filter(F.col("active_flag") == F.lit("Y"))
)

In [0]:
# sales_orders
sales_orders = (
    sales_orders
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("payment_method", F.upper(F.trim(F.col("payment_method"))))
    .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn("sales_channel", F.upper(F.trim(F.col("sales_channel"))))
    .withColumn("warehouse_id", F.trim(F.col("warehouse_id")))
    .withColumn("order_timestamp", F.to_timestamp(F.col("order_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("quantity", F.col("quantity").try_cast(IntegerType()))
    .withColumn("discount_pct", F.col("discount_pct").try_cast(DecimalType(5, 2)))
    .withColumn("promised_delivery_date", F.to_date(F.col("promised_delivery_date"), "yyyy-MM-dd"))
    .withColumn("actual_delivery_date", F.to_date(F.col("actual_delivery_date"), "yyyy-MM-dd"))
    .withColumn("order_month", F.date_format(F.col("order_timestamp"), "yyyy-MM"))
    .withColumn(
        "late_delivery_flag",
        F.when(F.col("actual_delivery_date").isNull(), None)
        .when(F.col("actual_delivery_date") > F.col("promised_delivery_date"), F.lit("Y"))
        .otherwise(F.lit("N"))
    )
    .withColumn("delivery_days", F.datediff(F.col("actual_delivery_date"), F.col("order_timestamp")))
    .drop("source_file", "ingestion_timestamp")
    .filter(F.col("quantity") > 0)
    .filter(~(F.col("order_status").isin(["CANCELLED", "PENDING"])))
)

In [0]:
# merge tables
full_table = (
    sales_orders.join(deduped_customers, "customer_id", "inner")
    .join(products, "product_id", "inner")
    .withColumn("gross_amount", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("discount_amount", F.round(F.col("gross_amount") * F.coalesce(F.col("discount_pct"), F.lit(0)) / 100, 2))
    .withColumn("net_amount", F.round(F.col("gross_amount") - F.col("discount_amount"), 2))
    .withColumn("net_sales", F.round(F.when(F.col("order_status") == "COMPLETED", F.col("net_amount")).otherwise(F.lit(0)), 2))
    .withColumn("profit_per_unit", F.round((F.col("net_amount") / F.col("quantity")) - F.col("cost_price"), 2))
)

In [0]:
# save silver tables
deduped_customers.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_customers")
products.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_products")
sales_orders.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_sales_orders")
full_table.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_full_table")

In [0]:
# read silver full table to create gold summary table
silver_table = spark.read.table("workspace.retail_fresher.silver_full_table")

In [0]:
# monthly category sales
monthly_cat_sales = (
    silver_table
    .filter(F.col("order_status") == "COMPLETED")
    .select("order_month", "category", "net_sales", "order_id", "quantity")
    .groupBy("order_month", "category")
    .agg(
        F.sum("net_sales").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
        F.sum(F.col("quantity")).alias("total_quantity")
    )
    .orderBy("order_month", "category")
    .withColumnRenamed("order_month", "sales_month")
)

# table is event format
events_table = (
    monthly_cat_sales
    .withColumn("event_id", F.concat(F.lit("SALES-"), F.col("sales_month"), F.lit("-"), F.col("category")))
    .withColumn("event_type", F.lit("MONTHLY_CATEGORY_SALES_READY"))
    .select("event_id", "event_type", "sales_month", "category", "order_count", "total_quantity", "total_revenue")
)

events_table.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_events")